[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/45_per_group_linear_solution.ipynb)

# 🔴 Solution: Per-Group Linear (MoE Forward)

**Primitive: fancy indexing + `bmm` (batched matrix multiply)**

**Reduction:** `y[i] = W[group_ids[i]] @ x[i] + b[group_ids[i]]`.

Key steps:
1. `W[group_ids]` — fancy indexing gathers the right `(d_out, d_in)` matrix for each token → shape `(N, d_out, d_in)`
2. `x.unsqueeze(-1)` → `(N, d_in, 1)` — column vectors
3. `torch.bmm(W_per, x_col).squeeze(-1)` → `(N, d_out)` — batched matrix-vector multiply
4. `+ b[group_ids]` — fancy indexing again for the bias

An equivalent formulation: `torch.einsum('noi,ni->no', W[group_ids], x) + b[group_ids]`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def per_group_linear(
    x: torch.Tensor,
    group_ids: torch.Tensor,
    W: torch.Tensor,
    b: torch.Tensor,
) -> torch.Tensor:
    # primitive: fancy indexing gathers per-token weights; bmm does batched mat-vec multiply
    W_per = W[group_ids]                                    # (N, d_out, d_in)
    out = torch.bmm(W_per, x.unsqueeze(-1)).squeeze(-1)     # (N, d_out)
    return out + b[group_ids]                               # (N, d_out)

In [ ]:
# Verify
x         = torch.tensor([[1.,0.],[0.,1.],[1.,1.]])
group_ids = torch.tensor([0, 1, 0])
W = torch.zeros(2, 2, 2)
W[0] = torch.eye(2)
W[1] = 2 * torch.eye(2)
b = torch.zeros(2, 2)
b[1] = torch.tensor([1., 1.])

result = per_group_linear(x, group_ids, W, b)
print('output:', result.tolist())
print('expect: [[1.0, 0.0], [1.0, 3.0], [1.0, 1.0]]')

# Gradient check
W2 = W.clone().requires_grad_(True)
b2 = b.clone().requires_grad_(True)
per_group_linear(x, group_ids, W2, b2).sum().backward()
print('W.grad:', W2.grad)
print('b.grad:', b2.grad)

In [ ]:
# Run judge
from torch_judge import check
check("per_group_linear")